# ChEMBL Webresource Client — API Reference

The  library provides Python access to ChEMBL data without needing SQL or REST knowledge. Results are cached locally so repeated calls are fast.

**Sections:**
1. Compounds
2. Activities
3. Assays
4. Tissues
5. Cell Lines
6. Targets
7. References / Documents
8. Sources
9. Utils
10. Protein Classification
11. Substructure Search
12. ATC Classification
13. Metabolism
14. Target Component
15. Mechanism of Action

## Available data entities

List all available API resources:

In [1]:
from chembl_webresource_client.new_client import new_client

available_resources = [resource for resource in dir(new_client) if not resource.startswith('_')]
print(available_resources)

['activity', 'activity_supplementary_data_by_activity', 'assay', 'assay_class', 'atc_class', 'binding_site', 'biotherapeutic', 'cell_line', 'chembl_id_lookup', 'chembl_release', 'compound_record', 'compound_structural_alert', 'description', 'document', 'document_similarity', 'drug', 'drug_indication', 'drug_warning', 'go_slim', 'image', 'mechanism', 'metabolism', 'molecule', 'molecule_form', 'official', 'organism', 'protein_classification', 'similarity', 'source', 'substructure', 'target', 'target_component', 'target_relation', 'tissue', 'xref_source']


## Available filters

The design of the client is based on Django QuerySet (https://docs.djangoproject.com/en/1.11/ref/models/querysets) and most important lookup types are supported:

- exact — e.g. max_phase=4
- iexact — case-insensitive exact, e.g. pref_name__iexact='aspirin'
- contains — e.g. description__contains='kinase'
- icontains — case-insensitive contains, e.g. description__icontains='kinase'
- in — e.g. molecule_chembl_id__in=['CHEMBL25', 'CHEMBL192']
- gt — greater than, e.g. year__gt=2020
- gte — greater than or equal, e.g. first_approval__gte=2000
- lt — less than, e.g. mw_freebase__lt=500
- lte — less than or equal, e.g. molecule_properties__mw_freebase__lte=300
- startswith — e.g. pref_name__startswith='imi'
- istartswith — case-insensitive, e.g. pref_name__istartswith='blood'
- endswith — e.g. pref_name__endswith='nib'
- iendswith — case-insensitive, e.g. pref_name__iendswith='nib'
- isnull — e.g. pchembl_value__isnull=False
- iregex — regular expression, e.g. description__iregex='nephrotoxicity|renal toxicity'

Nested fields use double underscores to traverse objects: molecule_properties__mw_freebase__lte=300 accesses mw_freebase inside molecule_properties.

## `only` operator

`only(['field1', 'field2'])` limits the response to specified fields — reduces payload size. The argument must be a list. Specifying a nested field name (e.g. `molecule_properties__alogp`) returns the entire parent object (`molecule_properties`); you must unpack it in code.

# 1. Compounds

Compounds in ChEMBL have associated bioactivity data measured against targets. Use `new_client.molecule` for all compound lookups. Retrieve by name, synonym, ChEMBL ID, InChI key, SMILES connectivity, structural similarity, or molecular property filters.

## 1.1 Find a compound by preferred name

Uses `pref_name__iexact` for a case-insensitive exact match on the preferred name field.

In [2]:
from chembl_webresource_client.new_client import new_client

molecule = new_client.molecule
mols = molecule.filter(pref_name__iexact='aspirin')
mols

[{'atc_classifications': ['B01AC06', 'N02BA01', 'N02BA51', 'A01AD05', 'N02BA71'], 'availability_type': 2, 'biotherapeutic': None, 'black_box_warning': 0, 'chebi_par_id': 15365, 'chirality': 2, 'cross_references': [{'xref_id': 'aspirin', 'xref_name': 'aspirin', 'xref_src': 'DailyMed'}, {'xref_id': '144203627', 'xref_name': 'SID: 144203627', 'xref_src': 'PubChem'}, {'xref_id': '144209315', 'xref_name': 'SID: 144209315', 'xref_src': 'PubChem'}, {'xref_id': '144210466', 'xref_name': 'SID: 144210466', 'xref_src': 'PubChem'}, {'xref_id': '170465039', 'xref_name': 'SID: 170465039', 'xref_src': 'PubChem'}, {'xref_id': '17389202', 'xref_name': 'SID: 17389202', 'xref_src': 'PubChem'}, {'xref_id': '17390036', 'xref_name': 'SID: 17390036', 'xref_src': 'PubChem'}, {'xref_id': '174007205', 'xref_name': 'SID: 174007205', 'xref_src': 'PubChem'}, {'xref_id': '26747283', 'xref_name': 'SID: 26747283', 'xref_src': 'PubChem'}, {'xref_id': '26752858', 'xref_name': 'SID: 26752858', 'xref_src': 'PubChem'}, {'

## 1.2 Find a compound by synonym

When a compound is better known by a trade name or synonym than its `pref_name`, filter on the nested `molecule_synonyms__molecule_synonym` field. The first `__` traverses into the `molecule_synonyms` list; the second `__` accesses the `molecule_synonym` string within it.

In [3]:
from chembl_webresource_client.new_client import new_client

molecule = new_client.molecule
mols = molecule.filter(molecule_synonyms__molecule_synonym__iexact='viagra').only('molecule_chembl_id')
mols

[{'molecule_chembl_id': 'CHEMBL192'}, {'molecule_chembl_id': 'CHEMBL1737'}]

## 1.3 Get a compound by ChEMBL ID

All ChEMBL entities have a stable ChEMBL ID. Filter with `chembl_id=` (or equivalently `molecule_chembl_id=`) and use `only` to select specific fields.

In [4]:
from chembl_webresource_client.new_client import new_client

molecule = new_client.molecule
m1 = molecule.filter(chembl_id='CHEMBL192').only(['molecule_chembl_id', 'pref_name', 'molecule_structures'])
m1

[{'molecule_chembl_id': 'CHEMBL192', 'molecule_structures': {'canonical_smiles': 'CCCc1nn(C)c2c(=O)[nH]c(-c3cc(S(=O)(=O)N4CCN(C)CC4)ccc3OCC)nc12', 'molfile': '\n     RDKit          2D\n\n 33 36  0  0  0  0  0  0  0  0999 V2000\n    2.1000   -0.0042    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0\n    2.1000    0.7000    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0\n   -1.5375   -0.0042    0.0000 S   0  0  0  0  0  0  0  0  0  0  0  0\n    1.4917   -0.3667    0.0000 N   0  0  0  0  0  0  0  0  0  0  0  0\n    0.8792   -0.0042    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0\n    2.8042    0.9083    0.0000 N   0  0  0  0  0  0  0  0  0  0  0  0\n    1.4917    1.0625    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0\n    0.8792    0.6833    0.0000 N   0  0  0  0  0  0  0  0  0  0  0  0\n    3.2042    0.3458    0.0000 N   0  0  0  0  0  0  0  0  0  0  0  0\n    2.8042   -0.2417    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0\n    0.2875   -0.3750    0.0000 C   0  0  0  0  0  0  0  0  0  

## 1.4 Get multiple compounds by ChEMBL ID list

Use `molecule_chembl_id__in=[...]` to retrieve a batch of compounds in one call.

In [5]:
from chembl_webresource_client.new_client import new_client

molecule = new_client.molecule
mols = molecule.filter(molecule_chembl_id__in=['CHEMBL25', 'CHEMBL192', 'CHEMBL27']).only(['molecule_chembl_id', 'pref_name'])
mols

[{'molecule_chembl_id': 'CHEMBL25', 'pref_name': 'ASPIRIN'}, {'molecule_chembl_id': 'CHEMBL27', 'pref_name': 'PROPRANOLOL'}, {'molecule_chembl_id': 'CHEMBL192', 'pref_name': 'SILDENAFIL'}]

## 1.5 Get a compound image (visualisation only)

The `new_client.image` endpoint returns SVG/PNG images. This is for display purposes — it does not return structured data. Use `image.get('CHEMBL_ID')` to retrieve the image bytes.

In [ ]:
from chembl_webresource_client.new_client import new_client
from IPython.display import SVG

image = new_client.image
image.set_format('svg')
SVG(image.get('CHEMBL25'))

## 1.6 Find a compound by standard InChI key

Filter on `molecule_structures__standard_inchi_key` to look up a compound by its InChI key. Returns the compound record including structure fields.

In [7]:
from chembl_webresource_client.new_client import new_client

molecule = new_client.molecule
mol = molecule.filter(molecule_structures__standard_inchi_key='BSYNRYMUTXBXSQ-UHFFFAOYSA-N').only(['molecule_chembl_id', 'pref_name', 'molecule_structures'])
mol

[{'molecule_chembl_id': 'CHEMBL25', 'molecule_structures': {'canonical_smiles': 'CC(=O)Oc1ccccc1C(=O)O', 'molfile': '\n     RDKit          2D\n\n 13 13  0  0  0  0  0  0  0  0999 V2000\n    8.8810   -2.1206    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0\n    8.8798   -2.9479    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0\n    9.5946   -3.3607    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0\n   10.3110   -2.9474    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0\n   10.3081   -2.1170    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0\n    9.5928   -1.7078    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0\n   11.0210   -1.7018    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0\n   11.7369   -2.1116    0.0000 O   0  0  0  0  0  0  0  0  0  0  0  0\n   11.0260   -3.3588    0.0000 O   0  0  0  0  0  0  0  0  0  0  0  0\n   11.0273   -4.1837    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0\n   11.7423   -4.5949    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0\n   10.3136   -4.5972    0.0000 O 

## 1.7 Similarity search by SMILES

Use `new_client.similarity` with `smiles=` and `similarity=` (integer 0–100, representing percentage) to find structurally similar compounds.

In [ ]:
from chembl_webresource_client.new_client import new_client

similarity = new_client.similarity
res = similarity.filter(smiles="CO[C@@H](CCC#C\C=C/CCCC(C)CCCCC=C)C(=O)[O-]", similarity=70).only(['molecule_chembl_id', 'similarity'])
for i in res:
    print(i)

## 1.8 Similarity search by ChEMBL ID

Alternative to SMILES: pass `chembl_id=` to search for compounds similar to a known molecule. Returns `molecule_chembl_id`, `pref_name`, and `similarity` score.

In [9]:
from chembl_webresource_client.new_client import new_client

similarity = new_client.similarity
res = similarity.filter(chembl_id='CHEMBL25', similarity=70).only(['molecule_chembl_id', 'pref_name', 'similarity'])
res

[{'molecule_chembl_id': 'CHEMBL2296002', 'pref_name': None, 'similarity': '100'}, {'molecule_chembl_id': 'CHEMBL1697753', 'pref_name': 'ASPIRIN DL-LYSINE', 'similarity': '100'}, {'molecule_chembl_id': 'CHEMBL3833325', 'pref_name': 'CARBASPIRIN CALCIUM', 'similarity': '88.8888895511627197265625'}, {'molecule_chembl_id': 'CHEMBL3833404', 'pref_name': 'CARBASPIRIN', 'similarity': '88.8888895511627197265625'}, '...(remaining elements truncated)...']

## 1.9 Find compounds with the same connectivity (SMILES)

Use `molecule_structures__canonical_smiles__connectivity=` to find all compounds sharing the same core connectivity as the given SMILES, regardless of stereochemistry or salt/solvate variations.

In [10]:
from chembl_webresource_client.new_client import new_client

molecule = new_client.molecule
res = molecule.filter(molecule_structures__canonical_smiles__connectivity='CN(C)C(=N)N=C(N)N').only(['molecule_chembl_id', 'pref_name'])
for i in res:
    print(i)

{'molecule_chembl_id': 'CHEMBL1431', 'pref_name': 'METFORMIN'}
{'molecule_chembl_id': 'CHEMBL1703', 'pref_name': 'METFORMIN HYDROCHLORIDE'}
{'molecule_chembl_id': 'CHEMBL3094198', 'pref_name': None}


## 1.10 Get all approved drugs

`max_phase=4` returns approved/marketed drugs. Use `order_by('molecule_properties__mw_freebase')` to sort by molecular weight ascending.

In [11]:
from chembl_webresource_client.new_client import new_client

molecule = new_client.molecule
approved_drugs = molecule.filter(max_phase=4).order_by('molecule_properties__mw_freebase')
approved_drugs

[{'atc_classifications': ['V03AN03'], 'availability_type': 1, 'biotherapeutic': None, 'black_box_warning': 0, 'chebi_par_id': 30217, 'chirality': 2, 'cross_references': [], 'dosed_ingredient': True, 'first_approval': 2015, 'first_in_class': 0, 'helm_notation': None, 'indication_class': 'Gases, Diluent for', 'inorganic_flag': 1, 'max_phase': 4, 'molecule_chembl_id': 'CHEMBL1796997', 'molecule_hierarchy': {'molecule_chembl_id': 'CHEMBL1796997', 'parent_chembl_id': 'CHEMBL1796997'}, 'molecule_properties': {'alogp': None, 'aromatic_rings': None, 'cx_logd': None, 'cx_logp': None, 'cx_most_apka': None, 'cx_most_bpka': None, 'full_molformula': 'He', 'full_mwt': '4.00', 'hba': None, 'hba_lipinski': None, 'hbd': None, 'hbd_lipinski': None, 'heavy_atoms': None, 'molecular_species': None, 'mw_freebase': '4.00', 'mw_monoisotopic': '4.0026', 'num_lipinski_ro5_violations': None, 'num_ro5_violations': None, 'psa': None, 'qed_weighted': None, 'ro3_pass': None, 'rtb': None}, 'molecule_structures': {'ca

## 1.11 Get drugs indicated for a disease

Two-step pattern: query `drug_indication` filtered by `efo_term__icontains` to get matching molecule IDs, then retrieve those molecules. The `drug_indication` endpoint uses EFO (Experimental Factor Ontology) disease terms.

In [12]:
from chembl_webresource_client.new_client import new_client

drug_indication = new_client.drug_indication
molecules = new_client.molecule

lung_cancer_ind = drug_indication.filter(efo_term__icontains="LUNG CARCINOMA")
lung_cancer_mols = molecules.filter(
    molecule_chembl_id__in=[x['molecule_chembl_id'] for x in lung_cancer_ind])

len(lung_cancer_mols)

631

## 1.12 Filter drugs by approval year and name stem using the `drug` endpoint

The `drug` endpoint (distinct from `molecule`) contains curated drug-specific fields: `first_approval` (year), `usan_stem` / `usan_stem_definition`, `development_phase`. Multiple `.filter()` calls can be chained — each narrows the result set further.

In [13]:
from chembl_webresource_client.new_client import new_client

drug = new_client.drug
res = drug.filter(first_approval__gte=1980).filter(usan_stem="-azosin")
res

[{'applicants': ['Hikma Pharmaceuticals International Ltd', 'Jubilant Cadista Pharmaceuticals Inc', 'Apnar Pharma Lp', 'Abbott Laboratories Pharmaceutical Products Div', 'Ranbaxy Laboratories Ltd', 'Beximco Pharmaceuticals Usa Inc', 'Ivax Pharmaceuticals Inc Sub Teva Pharmaceuticals Usa', 'Mylan Technologies Inc', 'Teva Pharmaceuticals Usa Inc', 'Sandoz Inc'], 'atc_classification': [{'code': 'G04CA03', 'description': 'GENITO URINARY SYSTEM AND SEX HORMONES: UROLOGICALS: DRUGS USED IN BENIGN PROSTATIC HYPERTROPHY: Alpha-adrenoreceptor antagonists'}], 'availability_type': 1, 'biotherapeutic': None, 'black_box': False, 'black_box_warning': '0', 'chirality': 0, 'development_phase': 4, 'drug_type': 1, 'first_approval': 1987, 'first_in_class': False, 'helm_notation': None, 'indication_class': 'Antihypertensive', 'molecule_chembl_id': 'CHEMBL611', 'molecule_properties': {'alogp': '1.06', 'aromatic_rings': 2, 'cx_logd': '0.95', 'cx_logp': '1.18', 'cx_most_apka': None, 'cx_most_bpka': '7.24', '

## 1.13 Get all biotherapeutic compounds

Filter `biotherapeutic__isnull=False` to find compounds that have biotherapeutic data (biologics, antibodies, peptides, etc.).

In [14]:
from chembl_webresource_client.new_client import new_client

molecule = new_client.molecule
biotherapeutics = molecule.filter(biotherapeutic__isnull=False)
len(biotherapeutics)

22963

## 1.14 Filter compounds by molecular weight

`molecule_properties__mw_freebase__lte=300` filters on the molecular weight of the free base form (nested inside `molecule_properties`).

In [15]:
from chembl_webresource_client.new_client import new_client

molecule = new_client.molecule
light_molecules = molecule.filter(molecule_properties__mw_freebase__lte=300)

len(light_molecules)

367682

## 1.15 Combined molecular weight and name filters

Multiple filter conditions in a single `.filter()` call act as AND. Here: MW ≤ 300 AND pref_name ends with 'nib' (kinase inhibitor suffix).

In [16]:
from chembl_webresource_client.new_client import new_client

molecule = new_client.molecule
light_nib_molecules = molecule.filter(molecule_properties__mw_freebase__lte=300, pref_name__iendswith="nib").only(['molecule_chembl_id', 'pref_name'])

light_nib_molecules

[{'molecule_chembl_id': 'CHEMBL276711', 'pref_name': 'SEMAXANIB'}, {'molecule_chembl_id': 'CHEMBL4594348', 'pref_name': 'ELSUBRUTINIB'}]

## 1.16 Filter by Lipinski Rule-of-Five violations

`molecule_properties__num_ro5_violations=0` returns drug-like compounds with no Lipinski violations.

In [17]:
from chembl_webresource_client.new_client import new_client

molecule = new_client.molecule
no_violations = molecule.filter(molecule_properties__num_ro5_violations=0)
len(no_violations)

1441706

# 2. Activities

The `activity` endpoint provides bioactivity measurements. Key fields:

- `standard_type` — measurement type: `IC50`, `Ki`, `EC50`, `Kd`, `potency`, etc.
- `standard_value` / `standard_units` — raw value and units (often nM)
- `pchembl_value` — standardised −log₁₀ value in molar; ≥5 (≤10 µM) is a typical hit threshold
- `assay_type` — `B` (binding), `F` (functional), `A` (ADMET), `T` (toxicity)
- `target_chembl_id` — the target being assayed
- `molecule_chembl_id` — the compound tested

**Note:** `target_chembl_id` only accepts exact ChEMBL IDs — it does not support `__icontains`. To filter by target name, first resolve the target ID using `new_client.target`.

## 2.1 Get all IC50 activities for a named target (two-step)

Resolve the target ChEMBL ID by name first (`target.filter(pref_name__iexact=...)`), then filter activities by that ID.

In [18]:
from chembl_webresource_client.new_client import new_client

target = new_client.target
activity = new_client.activity
herg = target.filter(pref_name__iexact='hERG').only('target_chembl_id')[0]
herg_activities = activity.filter(target_chembl_id=herg['target_chembl_id']).filter(standard_type="IC50")

len(herg_activities)

13200

## 2.2 Get binding activities for a target by ChEMBL ID

`assay_type='B'` restricts to binding assays. Other assay types: `F` (functional), `A` (ADMET), `T` (toxicity), `P` (physicochemical).

In [19]:
from chembl_webresource_client.new_client import new_client

activity = new_client.activity
res = activity.filter(target_chembl_id='CHEMBL3938', assay_type='B')

len(res)

860

## 2.3 Get all activities with a pChEMBL value for a compound

`pchembl_value__isnull=False` filters to standardised activity records only (excludes raw qualitative or unit-inconsistent measurements).

In [20]:
from chembl_webresource_client.new_client import new_client

activities = new_client.activity
res = activities.filter(molecule_chembl_id="CHEMBL25", pchembl_value__isnull=False)

len(res)

138

# 3. Assays

## 6.1 Find ADMET assays by description keyword

`assay_type='A'` restricts to ADMET assays. `description__icontains='inhibit'` filters on the assay free-text description. Add `assay_organism=` to further restrict by species (e.g. `'Rattus norvegicus'`).

In [21]:
from chembl_webresource_client.new_client import new_client
assay = new_client.assay
res = assay.filter(description__icontains='inhibit', assay_type='A')
res

[{'assay_category': None, 'assay_cell_type': None, 'assay_chembl_id': 'CHEMBL884521', 'assay_classifications': [], 'assay_organism': 'Rattus norvegicus', 'assay_parameters': [], 'assay_strain': None, 'assay_subcellular_fraction': None, 'assay_tax_id': 10116, 'assay_test_type': None, 'assay_tissue': None, 'assay_type': 'A', 'assay_type_description': 'ADME', 'bao_format': 'BAO_0000357', 'bao_label': 'single protein format', 'cell_chembl_id': None, 'confidence_description': 'Direct single protein target assigned', 'confidence_score': 9, 'description': 'Inhibition of cytochrome P450 progesterone 15-alpha hydroxylase', 'document_chembl_id': 'CHEMBL1125500', 'relationship_description': 'Direct protein target assigned', 'relationship_type': 'D', 'src_assay_id': None, 'src_id': 1, 'target_chembl_id': 'CHEMBL3705', 'tissue_chembl_id': None, 'variant_sequence': None}, {'assay_category': None, 'assay_cell_type': None, 'assay_chembl_id': 'CHEMBL615148', 'assay_classifications': [], 'assay_organism

# 4. Tissues

Tissues can be looked up by ontology ID (Uberon, BTO, Caloha, EFO) or by `pref_name`. Tissue records are linked to assays performed in that tissue.

## 4.1 Get tissue by BTO ID (Brenda Tissue Ontology)

In [22]:
from chembl_webresource_client.new_client import new_client

tissue = new_client.tissue
res = tissue.filter(bto_id="BTO:0001073")
res

[{'bto_id': 'BTO:0001073', 'caloha_id': 'TS-0798', 'efo_id': 'EFO:0000857', 'pref_name': 'Pituitary gland', 'tissue_chembl_id': 'CHEMBL3638173', 'uberon_id': 'UBERON:0000007'}]

## 4.2 Get tissue by Caloha ID

In [23]:
from chembl_webresource_client.new_client import new_client

tissue = new_client.tissue
res = tissue.filter(caloha_id="TS-0490")
res

[{'bto_id': 'BTO:0000648', 'caloha_id': 'TS-0490', 'efo_id': 'EFO:0000834', 'pref_name': 'Intestine', 'tissue_chembl_id': 'CHEMBL3638176', 'uberon_id': 'UBERON:0000160'}]

## 4.3 Get tissue by Uberon ID (cross-species anatomy ontology)

In [24]:
from chembl_webresource_client.new_client import new_client

tissue = new_client.tissue
res = tissue.filter(uberon_id="UBERON:0000173")
res

[{'bto_id': 'BTO:0000068', 'caloha_id': 'TS-0034', 'efo_id': None, 'pref_name': 'Amniotic fluid', 'tissue_chembl_id': 'CHEMBL3638177', 'uberon_id': 'UBERON:0000173'}]

## 4.4 Get tissue by name prefix

`pref_name__istartswith=` for a case-insensitive prefix match.

In [25]:
from chembl_webresource_client.new_client import new_client

tissue = new_client.tissue
res = tissue.filter(pref_name__istartswith='blood')
res

[{'bto_id': None, 'caloha_id': None, 'efo_id': None, 'pref_name': 'Blood brain barrier', 'tissue_chembl_id': 'CHEMBL3987461', 'uberon_id': 'UBERON:0000120'}, {'bto_id': 'BTO:0000089', 'caloha_id': 'TS-0079', 'efo_id': 'EFO:0000296', 'pref_name': 'Blood', 'tissue_chembl_id': 'CHEMBL3638178', 'uberon_id': 'UBERON:0000178'}, {'bto_id': 'BTO:0001102', 'caloha_id': 'TS-0080', 'efo_id': 'EFO:0000817', 'pref_name': 'Blood vessel', 'tissue_chembl_id': 'CHEMBL3987656', 'uberon_id': 'UBERON:0001981'}, {'bto_id': 'BTO:0000102', 'caloha_id': None, 'efo_id': None, 'pref_name': 'Blood clot', 'tissue_chembl_id': 'CHEMBL3987655', 'uberon_id': 'UBERON:0010210'}, '...(remaining elements truncated)...']

# 5. Cell Lines

Cell lines can be looked up by Cellosaurus ID or by `cell_description`. Cell line records are linked to assays performed in that cell line.

## 5.1 Get a cell line by Cellosaurus ID

Cellosaurus (https://www.cellosaurus.org) is the standard registry for cell lines.

In [26]:
from chembl_webresource_client.new_client import new_client

cell_line = new_client.cell_line
res = cell_line.filter(cellosaurus_id="CVCL_0417")
res

[{'cell_chembl_id': 'CHEMBL3307686', 'cell_description': 'MDA-MB-435 (Breast metastasis of melanoma cells', 'cell_id': 687, 'cell_name': 'MDA-MB-435', 'cell_source_organism': 'Homo sapiens', 'cell_source_tax_id': 9606, 'cell_source_tissue': 'Breast metastasis of melanoma cells', 'cellosaurus_id': 'CVCL_0417', 'cl_lincs_id': None, 'clo_id': None, 'efo_id': 'EFO_0001213'}]

# 6. Targets

Targets represent biological entities being assayed — most commonly single proteins, but also complexes, cell lines, and organisms. Key fields: `target_chembl_id`, `pref_name`, `target_type`, `organism`.

Lookup patterns:
- By gene name: `target.filter(target_synonym__icontains='EGFR')`
- By name fragment: `target.filter(pref_name__icontains='kinase')`
- By UniProt accession: `target.filter(target_components__accession='P00533')`
- By protein family: use `new_client.protein_classification` to get class IDs first (see section 11)

## 6.1 Find a target by gene name

`target_synonym__icontains` searches the target synonym list, which includes gene names, UniProt names, and aliases. Use `only` to return a subset of fields.

In [1]:
from chembl_webresource_client.new_client import new_client

target = new_client.target
gene_name = 'BRD4'
res = target.filter(target_synonym__icontains=gene_name).only(['organism', 'pref_name', 'target_type'])
for i in res:
    print(i)

{'organism': 'Homo sapiens', 'pref_name': 'Bromodomain-containing protein 4', 'target_type': 'SINGLE PROTEIN'}
{'organism': 'Mus musculus', 'pref_name': 'Bromodomain-containing protein 4', 'target_type': 'SINGLE PROTEIN'}
{'organism': 'Homo sapiens', 'pref_name': 'BRD4/HDAC1', 'target_type': 'PROTEIN COMPLEX'}
{'organism': 'Homo sapiens', 'pref_name': 'Protein cereblon/Cullin-4A/Bromodomain-containing protein 4', 'target_type': 'PROTEIN-PROTEIN INTERACTION'}
{'organism': 'Homo sapiens', 'pref_name': 'Protein cereblon/Bromodomain-containing protein 4', 'target_type': 'PROTEIN-PROTEIN INTERACTION'}
{'organism': 'Homo sapiens', 'pref_name': 'von Hippel-Lindau disease tumor suppressor/Bromodomain-containing protein 4', 'target_type': 'PROTEIN-PROTEIN INTERACTION'}
{'organism': 'Homo sapiens', 'pref_name': 'Protein cereblon/DNA damage-binding protein 1/Bromodomain-containing protein 4', 'target_type': 'PROTEIN-PROTEIN INTERACTION'}
{'organism': 'Homo sapiens', 'pref_name': 'von Hippel-Linda

# 7. References / Documents

Document records map to scientific papers, patents, and datasets containing the bioactivity measurements in ChEMBL. Filter by `pubmed_id`, `year`, `doc_type` (`PUBLICATION`, `PATENT`, `DATASET`), or `journal`.

## 7.1 Find documents by PubMed ID list

Use `pubmed_id__in=(...)` to check which of a list of PubMed IDs exist in ChEMBL.

In [15]:
from chembl_webresource_client.new_client import new_client
ids = (27502541, 27584694, 27977190, 81377812)
pubmed_IDs = new_client.document
pm = pubmed_IDs.filter(pubmed_id__in=ids).only('pubmed_id')
pm

[{'pubmed_id': 27502541}, {'pubmed_id': 27584694}, {'pubmed_id': 27977190}]

## 7.2 Find datasets produced after a given year

`doc_type='DATASET'` restricts to dataset records. `year__gte=` filters by publication year.

In [16]:
datasets = new_client.document
ds = datasets.filter(year__gte=2021, doc_type = 'DATASET')
ds

[{'abstract': '', 'authors': 'University of Dundee', 'doc_type': 'DATASET', 'document_chembl_id': 'CHEMBL3988442', 'doi': '10.6019/CHEMBL3988442', 'doi_chembl': None, 'first_page': None, 'issue': None, 'journal': None, 'journal_full_title': None, 'last_page': None, 'patent_id': None, 'pubmed_id': None, 'src_id': 33, 'title': 'University of Dundee, Small-Polar-MMV Screening Library', 'volume': None, 'year': 2021}, {'abstract': 'SGC Frankfurt donated chemical probe project: A-079 was donated by Abbvie. Website: https://www.sgc-ffm.uni-frankfurt.de/#!specificprobeoverview/A-079. Control: A-226. References: 1. Bianchi, Bruce R, Xu-Feng Zhang, Regina M Reilly, Philip R Kym, Betty B Yao, and Jun Chen. 2012. ‘Species Comparison and Pharmacological Characterization of Human, Monkey, Rat, and Mouse TRPA1 Channels’. The Journal of Pharmacology and Experimental Therapeutics 341(2):360–68. PMID: 22319196. 2. Chen, Jun, Shailen K Joshi, Stanley DiDomenico, Richard J Perner, Joe P Mikusa, Donna M Ga

# 8. Sources

Source records describe the external databases and assay sources from which ChEMBL data was curated (e.g. BindingDB, PubChem BioAssay).

## 8.1 Get all ChEMBL data sources

`new_client.source` — no filter needed; returns all sources.

In [17]:
sources = new_client.source
sources

[{'src_description': 'Undefined', 'src_id': 0, 'src_short_name': 'UNDEFINED'}, {'src_description': 'Scientific Literature', 'src_id': 1, 'src_short_name': 'LITERATURE'}, {'src_description': 'GSK Malaria Screening', 'src_id': 2, 'src_short_name': 'GSK_TCMDC'}, {'src_description': 'Novartis Malaria Screening', 'src_id': 3, 'src_short_name': 'NOVARTIS'}, '...(remaining elements truncated)...']

# 9. Utils

Cheminformatics utilities for structure conversion, descriptor calculation, and standardisation. Uses `chembl_webresource_client.utils.utils` (not `new_client`).

## 9.1 Convert SMILES to CTAB (molblock)

`utils.smiles2ctab(smiles)` — required input for other utils functions.

In [28]:
from chembl_webresource_client.utils import utils

aspirin = utils.smiles2ctab('O=C(Oc1ccccc1C(=O)O)C')
aspirin

'\n     RDKit          2D\n\n 13 13  0  0  0  0  0  0  0  0999 V2000\n   -0.9550   -1.3220    0.0000 O   0  0  0  0  0  0  0  0  0  0  0  0\n   -0.9528   -0.3220    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0\n   -0.0858    0.1764    0.0000 O   0  0  0  0  0  0  0  0  0  0  0  0\n    0.7792   -0.3254    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0\n    0.7772   -1.3254    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0\n    1.6422   -1.8270    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0\n    2.5092   -1.3288    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0\n    2.5112   -0.3288    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0\n    1.6462    0.1728    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0\n    1.6482    1.1728    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0\n    0.7832    1.6746    0.0000 O   0  0  0  0  0  0  0  0  0  0  0  0\n    2.5152    1.6710    0.0000 O   0  0  0  0  0  0  0  0  0  0  0  0\n   -1.8178    0.1798    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0\n  1  2 

## 9.2 Compute Maximal Common Substructure (MCS)

`utils.mcs(smiles_list)` — returns the MCS of a list of SMILES.

In [29]:
from chembl_webresource_client.utils import utils

smiles = ["O=C(NCc1cc(OC)c(O)cc1)CCCC/C=C/C(C)C",
          "CC(C)CCCCCC(=O)NCC1=CC(=C(C=C1)O)OC", "c1(C=O)cc(OC)c(O)cc1"]
mols = [utils.smiles2ctab(smile) for smile in smiles]
sdf = ''.join(mols)
result = utils.mcs(sdf)
result

'[#6]1(-[#6]):[#6]:[#6](-[#8]-[#6]):[#6](:[#6]:[#6]:1)-[#8]'

## 9.3 Compute molecular descriptors from CTAB

`utils.chemblDescriptors(ctab)` — returns a JSON string of descriptor values.

In [30]:
from chembl_webresource_client.utils import utils
import json

aspirin = utils.smiles2ctab('O=C(Oc1ccccc1C(=O)O)C')
descs = json.loads(utils.chemblDescriptors(aspirin))[0]
descs

{'qed': 0.5501217966938848,
 'MolWt': 180.15899999999996,
 'TPSA': 63.60000000000001,
 'HeavyAtomCount': 13,
 'NumAromaticRings': 1,
 'NumHAcceptors': 3,
 'NumHDonors': 1,
 'NumRotatableBonds': 2,
 'MolLogP': 1.3100999999999998,
 'MolecularFormula': 'C9H8O4',
 'Ro3Pass': 0,
 'NumRo5': 0,
 'MonoisotopicMolWt': 180.042258736}

## 9.4 Compute structural alerts

`utils.structuralAlerts(ctab)` — flags known problematic substructures (PAINS, etc.).

In [31]:
from chembl_webresource_client.utils import utils

mol = utils.smiles2ctab("O=C(Oc1ccccc1C(=O)O)C")
alerts = json.loads(utils.structuralAlerts(mol))
for a in alerts[0]:
    print(a)

{'alert_id': 1030, 'alert_name': 'Ester', 'set_name': 'MLSMR', 'smarts': '[#6]-C(=O)O-[#6]'}
{'alert_id': 1069, 'alert_name': 'vinyl michael acceptor1', 'set_name': 'MLSMR', 'smarts': '[#6]-[CH1]=C-C(=O)[#6,#7,#8]'}


## 9.5 Standardise a molecule

`utils.standardize(ctab)` — returns a standardised molblock (neutralise, remove salts, etc.).

In [32]:
from chembl_webresource_client.utils import utils
mol = utils.smiles2ctab("[Na]OC(=O)Cc1ccc(C[NH3+])cc1.c1nnn[n-]1.O")
st = json.loads(utils.standardize(mol))
st

[{'standard_molblock': '\n     RDKit          2D\n\n 19 17  0  0  0  0  0  0  0  0999 V2000\n    0.0000   -3.0000    0.0000 O   0  0  0  0  0  0  0  0  0  0  0  0\n   -5.5170   -1.9538    0.0000 Na  0  0  0  0  0 15  0  0  0  0  0  0\n   -2.9244   -0.4442    0.0000 O   0  0  0  0  0  0  0  0  0  0  0  0\n   -2.0602    0.0590    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0\n   -2.0638    1.0590    0.0000 O   0  0  0  0  0  0  0  0  0  0  0  0\n   -1.1924   -0.4380    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0\n   -0.3282    0.0652    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0\n    0.5396   -0.4318    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0\n    1.4038    0.0714    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0\n    1.4002    1.0714    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0\n    2.2644    1.5744    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0\n    2.2608    2.5744    0.0000 N   0  0  0  0  0  0  0  0  0  0  0  0\n    0.5324    1.5682    0.0000 C   0  0  0  0  0  0  0 

## 9.6 Get the parent molecule

`utils.getParent(ctab)` — strips salts and returns the parent structure as a molblock.

In [33]:
from chembl_webresource_client.utils import utils

mol = utils.smiles2ctab("[Na]OC(=O)Cc1ccc(C[NH3+])cc1.c1nnn[n-]1.[Na]")
par = json.loads(utils.getParent(mol))
par

[{'parent_molblock': '\n     RDKit          2D\n\n 18 18  0  0  0  0  0  0  0  0999 V2000\n   -5.5170   -1.9538    0.0000 Na  0  0  0  0  0  1  0  0  0  0  0  0\n   -2.9244   -0.4442    0.0000 O   0  0  0  0  0  0  0  0  0  0  0  0\n   -2.0602    0.0590    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0\n   -2.0638    1.0590    0.0000 O   0  0  0  0  0  0  0  0  0  0  0  0\n   -1.1924   -0.4380    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0\n   -0.3282    0.0652    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0\n    0.5396   -0.4318    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0\n    1.4038    0.0714    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0\n    1.4002    1.0714    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0\n    2.2644    1.5744    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0\n    2.2608    2.5744    0.0000 N   0  0  0  0  0  0  0  0  0  0  0  0\n    0.5324    1.5682    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0\n   -0.3318    1.0652    0.0000 C   0  0  0  0  0  0  0  0

# 10. Protein Classification

Use `new_client.protein_classification` to retrieve the ChEMBL protein class hierarchy (up to 8 levels: `l1`..`l8`) and find targets belonging to a given family.

**Important:** the resource name is `protein_classification`, NOT `protein_class`.

Key fields: `protein_class_id`, `pref_name`, `protein_class_desc`, `l1`..`l6`.

Typical two-step pattern to find all targets in a family:
1. Get `protein_class_id` values for the family using `protein_classification.filter(l1__icontains=...)`
2. Filter targets: `target.filter(target_components__protein_classifications__protein_class_id__in=[...])`

In [ ]:
from chembl_webresource_client.new_client import new_client
import pandas as pd

protein_classification = new_client.protein_classification

# Find all kinase classes (l2 is the second level of the hierarchy)
kinase_classes = protein_classification.filter(l2__icontains='kinase')
df = pd.DataFrame(kinase_classes)
df

In [ ]:
from chembl_webresource_client.new_client import new_client
import pandas as pd

protein_classification = new_client.protein_classification
target = new_client.target

# Step 1: get protein_class_id values for GPCRs
gpcr_classes = protein_classification.filter(l1__icontains='gpcr').only(
    ['protein_class_id', 'pref_name', 'protein_class_desc'])
gpcr_class_ids = [str(x['protein_class_id']) for x in gpcr_classes]

# Step 2: find human targets belonging to those classes
gpcr_targets = target.filter(
    target_components__protein_classifications__protein_class_id__in=gpcr_class_ids,
    organism='Homo sapiens'
).only(['target_chembl_id', 'pref_name', 'target_type', 'organism'])
df = pd.DataFrame(gpcr_targets)
df

# 11. Substructure Search

Use `new_client.substructure` to find all compounds containing a given SMILES substructure. Pass the substructure SMILES to the `smiles` filter. Note: these queries can be slow on the server for common fragments.

In [ ]:
from chembl_webresource_client.new_client import new_client
import pandas as pd

substructure = new_client.substructure

# Find all molecules containing a benzimidazole core
res = substructure.filter(smiles='c1ccc2[nH]cnc2c1').only(
    ['molecule_chembl_id', 'pref_name'])
df = pd.DataFrame(res)
df

In [ ]:
from chembl_webresource_client.new_client import new_client
import pandas as pd

substructure = new_client.substructure
molecule = new_client.molecule

# Find approved drugs containing a sulfonamide group
res = substructure.filter(smiles='NS(=O)(=O)c1ccccc1').only(
    ['molecule_chembl_id', 'pref_name', 'max_phase'])
df = pd.DataFrame(res)
df = df[df['max_phase'] == 4]  # approved only
df

# 12. ATC Classification

The WHO ATC (Anatomical Therapeutic Chemical) classification system. Use `new_client.atc_class` to retrieve ATC codes and descriptions.

Key fields: `level1`..`level5` (codes), `level1_description`..`level4_description`, `who_name`.

You can also filter the `molecule` endpoint directly using `atc_classifications__level1=` (etc.) to get all approved drugs in a given ATC class.

In [ ]:
from chembl_webresource_client.new_client import new_client
import pandas as pd

atc_class = new_client.atc_class

# Get all ATC class L entries (antineoplastic and immunomodulating agents)
res = atc_class.filter(level1='L')
df = pd.DataFrame(res)
df

In [ ]:
from chembl_webresource_client.new_client import new_client
import pandas as pd

# Filter approved drugs directly by ATC level using the molecule endpoint
molecule = new_client.molecule
res = molecule.filter(
    atc_classifications__level1='J',   # antiinfectives for systemic use
    max_phase=4                        # approved drugs only
).only(['molecule_chembl_id', 'pref_name', 'atc_classifications', 'max_phase'])
df = pd.DataFrame(res)
df

# 13. Metabolism

Use `new_client.metabolism` to retrieve drug metabolism records.

Key fields: `substrate_chembl_id` (the drug being metabolised), `metabolite_chembl_id`, `metabolite_name`, `enzyme_name`, `organism`, `target_chembl_id`.

**Filter by `substrate_chembl_id`** (not `drug_chembl_id`) to get metabolites of a drug.

In [ ]:
from chembl_webresource_client.new_client import new_client
import pandas as pd

metabolism = new_client.metabolism

# Get metabolism records for imatinib (CHEMBL941) as the substrate
res = metabolism.filter(substrate_chembl_id='CHEMBL941')
df = pd.DataFrame(res)
df

In [ ]:
from chembl_webresource_client.new_client import new_client
import pandas as pd

metabolism = new_client.metabolism

# Get all human CYP-mediated metabolism records
res = metabolism.filter(organism='Homo sapiens')
df = pd.DataFrame(res)
df

# 14. Target Component

Use `new_client.target_component` to retrieve protein sequence and annotation data for ChEMBL targets.

Key fields: `accession` (UniProt ID), `component_type`, `description`, `organism`.

Filter by `accession=` to find the ChEMBL target record associated with a UniProt protein.

In [ ]:
from chembl_webresource_client.new_client import new_client
import pandas as pd

target_component = new_client.target_component

# Look up EGFR by UniProt accession (P00533)
res = target_component.filter(accession='P00533')
df = pd.DataFrame(res)
df

# 15. Mechanism of Action

Use `new_client.mechanism` to retrieve curated drug mechanism of action records.

Key fields: `action_type` (e.g. INHIBITOR, ANTAGONIST, AGONIST), `mechanism_of_action` (text description), `molecule_chembl_id`, `target_chembl_id`, `max_phase`.

Filter by `target_chembl_id` (all drugs acting on a target) or `molecule_chembl_id` (mechanism for a specific drug).

In [ ]:
from chembl_webresource_client.new_client import new_client
import pandas as pd

mechanism = new_client.mechanism

# Get all drugs with a mechanism of action on EGFR (CHEMBL203)
res = mechanism.filter(target_chembl_id='CHEMBL203').only(
    ['action_type', 'mechanism_of_action', 'molecule_chembl_id', 'max_phase'])
df = pd.DataFrame(res)
df

In [ ]:
from chembl_webresource_client.new_client import new_client
import pandas as pd

mechanism = new_client.mechanism

# Get the mechanism of action for imatinib (CHEMBL941)
res = mechanism.filter(molecule_chembl_id='CHEMBL941')
df = pd.DataFrame(res)
df